# NDMS Persistence–Stabilization Demonstration

This notebook provides a reduced zero-dimensional demonstration of the transient nano-dense molecular state (NDMS) persistence–stabilization closure for combustion nanoparticle inception.

The purpose is to show how the formation of persistent incipient particles depends not only on precursor association, but also on the competition between dissociation, non-stabilizing losses, and chemical or structural stabilization.


## Core Equations

The bounded stabilization probability is

$$P_{\mathrm{stab}} = \frac{k_{\mathrm{stab}}}{k_{\mathrm{off}} + k_{\mathrm{loss}} + k_{\mathrm{stab}}}$$

The reduced quasi-steady NDMS inception source is

$$S_{\mathrm{NDMS}} = \frac{k_{\mathrm{stab}} k_{\mathrm{on}} \left(C_P^{\mathrm{assoc}}\right)^m}{k_{\mathrm{off}} + k_{\mathrm{loss}} + k_{\mathrm{stab}}}$$

For the simple demonstration below, the reservoir formation rate is set to

$$k_{\mathrm{on}} \left(C_P^{\mathrm{assoc}}\right)^m = 1$$


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def stabilization_probability(k_off, k_loss, k_stab):
    """Bounded stabilization probability."""
    denominator = k_off + k_loss + k_stab
    if np.any(denominator <= 0):
        raise ValueError("The denominator must be positive.")
    return k_stab / denominator


def ndms_source(k_on, c_assoc, m, k_off, k_loss, k_stab):
    """Reduced quasi-steady NDMS inception source."""
    formation_rate = k_on * c_assoc**m
    denominator = k_off + k_loss + k_stab
    if np.any(denominator <= 0):
        raise ValueError("The denominator must be positive.")
    return k_stab * formation_rate / denominator


## Demonstration Cases

The following cases illustrate three regimes:

1. **Rapid removal**: dissociation and loss dominate over stabilization.
2. **Comparable loss and stabilization**: stabilization competes with removal.
3. **Strong stabilization**: stabilization dominates the transient reservoir fate.


In [ ]:
cases = pd.DataFrame(
    [
        {
            "case": "Rapid removal",
            "k_on": 1.0,
            "C_assoc": 1.0,
            "m": 1.0,
            "k_off": 10.0,
            "k_loss": 1.0,
            "k_stab": 0.1,
        },
        {
            "case": "Comparable loss and stabilization",
            "k_on": 1.0,
            "C_assoc": 1.0,
            "m": 1.0,
            "k_off": 0.7,
            "k_loss": 0.3,
            "k_stab": 1.0,
        },
        {
            "case": "Strong stabilization",
            "k_on": 1.0,
            "C_assoc": 1.0,
            "m": 1.0,
            "k_off": 0.05,
            "k_loss": 0.05,
            "k_stab": 10.0,
        },
    ]
)

cases["P_stab"] = stabilization_probability(
    cases["k_off"], cases["k_loss"], cases["k_stab"]
)

cases["S_NDMS"] = ndms_source(
    cases["k_on"],
    cases["C_assoc"],
    cases["m"],
    cases["k_off"],
    cases["k_loss"],
    cases["k_stab"],
)

cases[["case", "k_off", "k_loss", "k_stab", "P_stab", "S_NDMS"]]


## Stabilization Probability as a Function of the Persistence–Stabilization Number

The persistence–stabilization number is

$$Da_{\mathrm{PS}} = \frac{k_{\mathrm{stab}}}{k_{\mathrm{off}} + k_{\mathrm{loss}}}$$

and the bounded probability can be written as

$$P_{\mathrm{stab}} = \frac{Da_{\mathrm{PS}}}{1 + Da_{\mathrm{PS}}}$$


In [ ]:
da_ps = np.logspace(-3, 3, 300)
p_stab = da_ps / (1.0 + da_ps)

plt.figure(figsize=(7, 5))
plt.semilogx(da_ps, p_stab, linewidth=2)
plt.xlabel("Persistence-stabilization number, Da_PS")
plt.ylabel("Bounded stabilization probability, P_stab")
plt.title("NDMS stabilization probability")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()


## Sensitivity to Stabilization Rate

The next example keeps the reservoir formation rate fixed and varies the stabilization coefficient. This shows how the NDMS inception source increases only when stabilization becomes competitive with dissociation and non-stabilizing loss.


In [ ]:
k_on = 1.0
c_assoc = 1.0
m = 1.0
k_off = 1.0
k_loss = 0.5
k_stab_values = np.logspace(-3, 2, 300)

sources = ndms_source(k_on, c_assoc, m, k_off, k_loss, k_stab_values)

plt.figure(figsize=(7, 5))
plt.semilogx(k_stab_values, sources, linewidth=2)
plt.xlabel("Stabilization coefficient, k_stab")
plt.ylabel("Reduced NDMS inception source, S_NDMS")
plt.title("Sensitivity of NDMS source to stabilization rate")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()


## Interpretation

This reduced demonstration illustrates the central modeling message of the NDMS persistence–stabilization closure:

- precursor association creates a transient reservoir,
- dissociation and non-stabilizing losses can remove that reservoir,
- only stabilization transfers part of the reservoir into persistent incipient particles,
- particle inception is therefore controlled by a competition between reversible clustering and persistence-generating stabilization.

The values used here are illustrative and dimensionless. They are not fitted parameters and should not be interpreted as validation data.
